# Agent memory - step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will demonstrate two AI agents sharing state in one governed Lakebase table. The agents read the shared memory, respond to questions, and append their turns - no message queue, no cache, just one table.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the factory is loaded on your branch. You need FM_ENDPOINT set to a running Foundation Model endpoint (default: databricks-claude-opus-4-8).

## Setup - initialize

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, _env, client

NAMESPACE = "default"
SESSION_ID = "shift-001"
FM_ENDPOINT = os.getenv("FM_ENDPOINT", "databricks-claude-opus-4-8")
TABLE = "factory.agent_memory"

print(f'Session: {SESSION_ID}, FM endpoint: {FM_ENDPOINT}')

## Step 1 - Seed shared memory

Seed the agent_memory table with a prior maintenance note. This is the shared state. The SQL:

```sql
INSERT INTO factory.agent_memory (namespace, session_id, agent_id, role, content, created_at)
VALUES ('default', 'shift-001', 'maintenance', 'system', 'M-204 showed high vibration yesterday, isolated a worn bearing');
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        # Clear prior memory (optional)
        cur.execute(f"DELETE FROM {TABLE} WHERE namespace=%s AND session_id=%s", [NAMESPACE, SESSION_ID])
        # Seed
        cur.execute(
            f"INSERT INTO {TABLE} (namespace, session_id, agent_id, role, content) "
            f"VALUES (%s, %s, %s, %s, %s)",
            [NAMESPACE, SESSION_ID, "maintenance", "user", "M-204 showed high vibration yesterday, isolated a worn bearing"]
        )
print('Seeded agent_memory with prior maintenance note')

## Step 2 - Read prior memory

Both agents read the shared memory before responding. The SQL:

```sql
SELECT agent_id, role, content FROM factory.agent_memory
WHERE namespace='default' AND session_id='shift-001'
ORDER BY created_at;
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"SELECT agent_id, role, content FROM {TABLE} "
            f"WHERE namespace=%s AND session_id=%s ORDER BY created_at",
            [NAMESPACE, SESSION_ID]
        )
        rows = cur.fetchall()

print(f'Prior memory ({len(rows)} rows):')
for agent_id, role, content in rows:
    print(f'  {agent_id}: {content[:80]}...' if len(content) > 80 else f'  {agent_id}: {content}')

## Step 3 - First agent responds and appends memory

The first agent (maintenance_advisor) reads the memory, responds to a question, and appends its turn. The SQL:

```sql
INSERT INTO factory.agent_memory (namespace, session_id, agent_id, role, content)
VALUES ('default', 'shift-001', 'maintenance_advisor', 'assistant', <response>);
```

In [ ]:
# Simulated response from maintenance advisor
response1 = "M-204 is cleared to run. The bearing replacement was thorough. Monitor vibration for the next 12 hours."

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"INSERT INTO {TABLE} (namespace, session_id, agent_id, role, content) "
            f"VALUES (%s, %s, %s, %s, %s)",
            [NAMESPACE, SESSION_ID, "maintenance_advisor", "assistant", response1]
        )

print(f'maintenance_advisor appended:')
print(f'  {response1}')

## Step 4 - Second agent reads full memory and responds

The second agent (mes_assistant) reads the FULL memory including the maintenance advisor's response, then appends its own turn.

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"SELECT agent_id, role, content FROM {TABLE} "
            f"WHERE namespace=%s AND session_id=%s ORDER BY created_at",
            [NAMESPACE, SESSION_ID]
        )
        full_memory = cur.fetchall()

print(f'Full memory for MES assistant ({len(full_memory)} rows):')
for agent_id, role, content in full_memory:
    print(f'  {agent_id}: {content[:70]}...' if len(content) > 70 else f'  {agent_id}: {content}')

# MES assistant's response
response2 = "M-204 scheduled for production run. Assigning WO-8901 (200 units) on the 0600 shift."

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"INSERT INTO {TABLE} (namespace, session_id, agent_id, role, content) "
            f"VALUES (%s, %s, %s, %s, %s)",
            [NAMESPACE, SESSION_ID, "mes_assistant", "assistant", response2]
        )

print(f'\nmes_assistant appended:')
print(f'  {response2}')

## Step 5 - Read final memory

Query the full conversation from the shared table.

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"SELECT agent_id, role, content FROM {TABLE} "
            f"WHERE namespace=%s AND session_id=%s ORDER BY created_at",
            [NAMESPACE, SESSION_ID]
        )
        final_memory = cur.fetchall()

print(f'Final memory ({len(final_memory)} total rows):')
for agent_id, role, content in final_memory:
    print(f'  [{agent_id}] {content}')

## What you learned

Two agents shared state in one governed Lakebase table without a message queue, cache, or orchestration. Each agent read the table, responded with context, and appended its turn. The pattern is: read, compute, append. Perfect for multi-agent coordination in manufacturing: maintenance advisor, MES, planning, quality control - all consulting the same table.

**Next:** Agentic work - agents with function calling, refusing duplicate work based on memory.